# Martinez-Conde et al. (2019) on CRCNS pvc-12

Paper: Martinez-Conde S, McCamy MB, Troncoso XG, Otero-Millan J, Macknik SL (2019). Area V1
responses to illusory corner-folds in Vasarely's nested squares and the Alternating Brightness Star
illusions. PLoS ONE 14(3): e0210941, doi:10.1371/journal.pone.0210941.

Dataset: CRCNS pvc-12, doi:10.6080/K05D8Q1T, read in place from the public bucket
(`pvc-12/live/V1_Vasarely_data.zip`). CRCNS data sets are not versioned, so the version is
recorded as `unversioned`; the only version there is is the one read. The zip's MD5 in the data
set's `checksums.md5` is `e1301fe463e72332e2303b2053ee0559` (printed again below from the bucket),
and its S3 Last-Modified date was 2026-02-11 when this attempt read it, so a later change can be
detected. The data descriptor read is version 0.7 (Feb 15, 2018).

The claims file (`claims/martinez-conde-2019.yaml`) was written from the version of record
(PMC6438452) with its figure legends; S1 and S2 Tables are not in the text read.

What the data set holds, in short: a spike rate map and the stimulus for each neuron and
experiment. The maps were made by the authors at their 60 ms latency. There are no spike times,
eye positions or eye time maps, apart from one neuron's eye time map. The analysis uses the maps
the authors registered to the stimulus (`Standard`), which exist for 22 Square and Edge, 26
Corners and 12 Star neurons.

| Claim | Tested here |
|---|---|
| neuron-counts | tested |
| spikes-per-condition | blocked (data_missing) |
| s4-baseline-no-difference | tested |
| fig4a-above-baseline | tested |
| fig4b-spike-rate-difference | tested |
| fig4b-pairwise-significance | tested |
| fig5-average-maps | tested |
| table1-edge, table1-p180, table1-s90, table1-s52, table1-s40 | tested |
| fig6-midpoint-representative | tested |
| fig7a-rms-contrast-fit | tested |
| fig7c-fold-edge-equivalence | tested |
| fig7c-fold-edge-significance | tested |
| fig7d-nonlinear | tested |
| fig7d-dog-similar | blocked (data_missing) |
| s3-latency-peak | blocked (data_missing) |

In [ ]:
import time, resource, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize
from statsmodels.stats.multitest import multipletests
import pvc12

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)
t0 = time.time()
print(urllib.request.urlopen(pvc12.CHECKSUMS).read().decode())
z = pvc12.open_zip()
recs = pvc12.load_all(z)
sheet = pvc12.read_sheet(z)
T = pvc12.measure(recs, "S")    # smoothed spike rate maps (main)
TU = pvc12.measure(recs, "I")   # unsmoothed maps (variant)
print(f"loaded in {time.time() - t0:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024:.0f} MB")

## Data check

The cell below prints what was loaded and checks the descriptor against the values. There are no
time series or trial tables in the data set, only maps already made by the authors, so the clock
check of the protocol cannot be run; in its place the cell shows that each registered map is the
neuron's screen map moved by one translation (and, for the Star, one rotation), and that the
stimulus luminance at the fold points used below is the one the paper describes.

In [ ]:
R = pd.DataFrame([{k: r.get(k) for k in ("folder", "code", "cell", "monkey", "stim_type", "has_standard",
                                          "reg_corr", "reg_offset", "star_rotation", "xf_err", "xf_angle",
                                          "base_file", "screen_rate_max", "screen_rate_nonint")} for r in recs])
print("Files (map + stimulus pairs) per folder:")
print(R.groupby("folder").agg(files=("code", "size"), cells=("cell", "nunique"), with_Standard=("has_standard", "sum")))
print("\nStimulus type per folder:")
print(R.groupby(["folder", "stim_type"]).size())
print("\nFields of the map files (distinct sets):")
for f, n in pd.Series([", ".join(r["fields"]) for r in recs]).value_counts().items():
    print(f"  {n:3d}  {f}")
print("\nMonkeys over all distinct cells:", R.drop_duplicates("cell").monkey.value_counts().to_dict())
print("Distinct cells over all folders:", R.cell.nunique())

print("\n--- units of the maps ---")
print("max of dat.image over files: min", R.screen_rate_max.min(), "max", R.screen_rate_max.max(),
      "| files where dat.image is not integer:", int(R.screen_rate_nonint.sum()), "of", int(R.screen_rate_nonint.notna().sum()))
for r in recs:
    if "eyetime_ms_total" in r:
        print(f"{r['code']}: has an eyetime map; total eye time {r['eyetime_ms_total'] / 1000:.0f} s; "
              f"image * eyetime is integer to {r['count_integer_err']:.1e}; spike count {r['spike_count_from_eyetime']:.0f}")
print("So dat.image is spikes per ms (count / eye time in ms), not raw counts as the descriptor says; maps x 1000 = spikes/s.")

print("\n--- spreadsheet against the files ---")
files_by_cell = R.groupby("cell").folder.apply(set)
for cond in ["Corners", "Star", "Edge-compare"]:
    s_cells = set(sheet.cell[sheet[cond]])
    f_cells = set(R.cell[R.folder == cond])
    print(f"{cond:13s} sheet {len(s_cells):2d}  files {len(f_cells):2d}  in sheet not files {sorted(s_cells - f_cells)}  in files not sheet {sorted(f_cells - s_cells)}")
print("sheet rows with no condition marked:", sheet.cell[~sheet[["Corners", "Star", "Edge-compare"]].any(axis=1)].tolist())

print("\n--- registration of the Standard maps ---")
for f, g in R[R.has_standard].groupby("folder"):
    print(f"{f:13s} n={len(g):2d}  correlation with the moved screen map: min {g.reg_corr.min():.4f} median {g.reg_corr.median():.4f}")
print("Edge and Corners polygon fit error (DU), max:", R.xf_err.max())
print("Star rotation of the ROI frame relative to the screen (deg), per neuron:",
      R[R.star_rotation.notna()].set_index("code").star_rotation.astype(int).to_dict())

print("\n--- ROI-frame geometry: frame shapes and fold midpoints (x, y in DU) per condition ---")
geo = []
for r in recs:
    if r["has_standard"] and r["stim_type"] != "CORNERS":
        g = pvc12.fold_geometry(r)
        for fd in ("Edge", "P180", "S90", "S52", "S40"):
            if fd in g:
                geo.append(dict(folder=r["folder"], shape=r["S"].shape, fold=fd, halves=len(g[fd]),
                                midpoints=tuple(tuple(np.round((h["p0"] + h["p100"]) / 2, 1)) for h in g[fd])))
print(pd.DataFrame(geo).groupby(["folder", "shape", "fold", "halves", "midpoints"]).size().to_string())
print("(six Corners frames are 16 DU smaller, the same geometry cropped by 8 DU on each side)")
print("\n--- stimulus luminance (%) at the fold points, from the stimulus image warped into the ROI frame ---")
lum = lambda r, p: float(pvc12.ndimage.map_coordinates(r["stim_roi"], [[p[1]], [p[0]]], order=1)[0])
rows = []
for r in recs:
    if r["has_standard"] and r["folder"] != "Star":
        g = pvc12.fold_geometry(r)
        for fd in ("Edge", "P180", "S90", "S40", "S40dark"):
            if fd in g:
                h = g[fd][0]
                off = np.array([2.0, 0.0]) if fd == "Edge" else np.zeros(2)  # just inside the rectangle
                rows.append(dict(fold=fd, l0=lum(r, h["p0"] + 0.03 * (h["p100"] - h["p0"]) + off),
                                 l50=lum(r, (h["p0"] + h["p100"]) / 2 + off),
                                 l100=lum(r, h["p100"] - 0.03 * (h["p100"] - h["p0"]) + off)))
print(pd.DataFrame(rows).groupby("fold").agg(["min", "max"]).round(1))
print("(points 3% in from each end; the Star stimulus is not warped, its geometry is taken from its polygons)")
ap = [pvc12.fold_geometry(r)["corners_apices"] for r in recs if r["folder"] == "Corners"]
print("Corners: neurons with exactly one bright and one dark 40-degree apex:",
      sum(sum(a["bright"] for a in x) == 1 and len(x) == 2 for x in ap), "of", len(ap))

print("\n--- maps used (spikes/s) ---")
for f in ["Edge-compare", "Star", "Corners"]:
    rs = [r for r in recs if r["folder"] == f and r["has_standard"]]
    print(f"{f:13s} smoothed map range {min(r['S'].min() for r in rs):.2f} to {max(r['S'].max() for r in rs):.1f}; "
          f"fraction of zero pixels per map {min((r['S'] == 0).mean() for r in rs):.2f} to {max((r['S'] == 0).mean() for r in rs):.2f}")
print("\nBaselines (spikes/s) by condition, main reading, leaving out zero pixels, and the files' BaseActivity:")
B = T.drop_duplicates("code")
print(B.groupby("folder")[["base_main", "base_nz", "base_file", "base_zero_frac"]].agg(["mean", "count"]).round(2))
print("\nNeurons analysed per fold:", T.groupby("fold").size().to_dict())

Observations from the check, used below:

- `dat.image` is a rate (spikes per ms); only one neuron (ABC050900J1A) carries an eye time map,
  from which its spike count can be recovered. The other neurons' spike counts cannot.
- The two files named `ABS...` in the `Star` folder hold Corners stimuli (`type` CORNERS) and
  their own maps; they are separate recordings of two Corners neurons, not Star recordings.
- The Star stimulus is drawn at the same orientation on the screen for every neuron. The diagonal
  aligned with the neuron's preferred orientation is known only through the rotation of the
  `Standard` frame, which 15 Star neurons lack, together with their receptive field offset; they
  cannot be registered and are left out. The Star analysis has 12 neurons.
- `Standard.BaseActivity`, present for 38 neurons, is not described and was not reproduced by any
  placement of 16 x 16 squares tried while preparing (not run here); it is shown as a variant only.

## neuron-counts

> Of 118 V1 neurons recorded from two monkeys (102 from monkey J, 16 from monkey A), 60 were
> discarded for technical problems and 58 analysed; 7 were tested in all three conditions, 11 in
> two and 40 in one, giving 25 neurons for Square and Edge, 30 for Star and 28 for Corners.

Neurons are identified by their code with the experiment prefix and run letter removed (the
spreadsheet uses the same codes). Counts are given with conditions taken from the folders, from
the stimulus type in the files (the two ABS files counted as Corners), and from the spreadsheet.

In [ ]:
def overlap_counts(cond_by_cell):
    n = cond_by_cell.apply(len)
    return {"neurons": int(len(n)), "Square and Edge": int(sum("Edge-compare" in s for s in cond_by_cell)),
            "Star": int(sum("Star" in s for s in cond_by_cell)), "Corners": int(sum("Corners" in s for s in cond_by_cell)),
            "all three": int((n == 3).sum()), "two": int((n == 2).sum()), "one": int((n == 1).sum())}

R["cond_stim"] = np.where(R.stim_type == "CORNERS", "Corners", R.folder)
sheet_sets = sheet.set_index("cell")[["Corners", "Star", "Edge-compare"]].apply(lambda row: {c for c in row.index if row[c]}, axis=1)
sheet_sets = sheet_sets[sheet_sets.apply(len) > 0]
counts = pd.DataFrame({
    "paper": {"neurons": 58, "Square and Edge": 25, "Star": 30, "Corners": 28, "all three": 7, "two": 11, "one": 40},
    "files, by folder": overlap_counts(R.groupby("cell").folder.apply(set)),
    "files, by stimulus": overlap_counts(R.groupby("cell").cond_stim.apply(set)),
    "spreadsheet": overlap_counts(sheet_sets),
    "registered maps (analysed here)": overlap_counts(R[R.has_standard].groupby("cell").cond_stim.apply(set)),
})
print(counts)
print("\nMonkeys among the distinct cells in the files:", R.drop_duplicates("cell").monkey.value_counts().to_dict(),
      "(paper: 102 recorded from J and 16 from A, of which 58 analysed; per-monkey analysed counts not given)")
print("Spreadsheet rows:", len(sheet), "of which with no condition marked:", int((sheet[["Corners", "Star", "Edge-compare"]].sum(axis=1) == 0).sum()))

**Result: not reproduced.** The files hold 53 distinct neurons (49 from monkey J, 4 from A),
against 58 analysed in the paper: 22 Square and Edge, 29 Star by folder (27 with a Star
stimulus) and 26 Corners, against 25, 30 and 28. The overlap counts are 7, 10 and 36 by folder (7,
8 and 38 by stimulus), against 7, 11 and 40. Only the count of neurons tested in all three
conditions matches; no per-condition count does. The spreadsheet lists the same 53 neurons plus 5
rows with no condition marked, and the descriptor says 53.

## spikes-per-condition (blocked)

> Each experimental condition required at least 10,000 spikes, and the mean number of spikes per
> condition was 35,998 (STD 25,981).

The spike count map is not in the data set: `dat.image` is spikes per ms (count divided by eye
time), and the eye time map is there for one neuron only. The cell prints that neuron's count.

In [ ]:
for r in recs:
    if "spike_count_from_eyetime" in r:
        print(r["folder"], r["code"], "spike count from image x eyetime:", round(r["spike_count_from_eyetime"]))
print("Files with an eye time map:", sum("eyetime_ms_total" in r for r in recs), "of", len(recs))

**Result: blocked (data_missing).** Spike counts per neuron and condition cannot be recovered
from rate maps without eye time; 1 of 77 files has an eye time map, and that neuron (Corners
ABC050900J1A) has 15,644 spikes on its map, above the 10,000 minimum, but one neuron cannot test
the mean and STD over 83 neuron-conditions.

## Measures shared by the claims below

Per neuron and fold: the absolute baseline (four 16 x 16 DU squares for Square and Edge and Star,
two for Corners; see `pvc12.py`), the midpoint fold response (8 x 8 DU square at 50% luminance,
two midpoints averaged for S90 and S52), and the spike rate difference (response minus baseline).
All values in spikes/s from the smoothed registered maps.

Where the baseline squares sit is not fixed by the text: for Square and Edge and the Star they are
"at the corners" of a square around the figure (centred on the corners, or inside them), and for
Corners they are two squares "in 0% luminance areas within the ROI" shown only in Fig 2C (readings
(a) the ROI's corners on the 0% side, (b) beside the 0% end of the bright fold, (c) on the ROI's
0% edge halfway between the fold and each side). The main reading for each stimulus is the one
whose values are closest, neuron by neuron, to the files' own `BaseActivity` (the authors'
baseline for 38 neurons, placement undocumented); the next cell prints that comparison. This is
an assumption, chosen after the results under every reading had been seen while preparing; every
reading is shown with each result.

In [ ]:
FOLDS = ["P180", "S90", "S52", "S40"]
ANGLE = {"P180": 180, "S90": 90, "S52": 52, "S40": 40}
def diffs(T, base="base_main"):
    d = T.copy()
    d["resp"] = d.mid_resp
    d["base"] = d[base]
    d["diff"] = d.mid_resp - d[base]
    return d
def with_base(T, edge=None, star=None, corners=None):
    d = T.copy()
    for f, col in (("Edge-compare", edge), ("Star", star), ("Corners", corners)):
        if col:
            d.loc[d.folder == f, "base_main"] = d.loc[d.folder == f, col]
    return d
VARIANTS = {"Edge squares inside the corners": dict(edge="base_inside"),
            "Star squares centred on the corners": dict(star="base_centred"),
            "Corners reading (b)": dict(corners="base_b_beside_fold"),
            "Corners reading (c)": dict(corners="base_c_roi_edge_mid")}
Bf = T.drop_duplicates("code")
Bf = Bf[Bf.base_file.notna()]
print("Baseline readings against the files' BaseActivity (spikes/s):")
for f, cols in [("Edge-compare", ["base_centred", "base_inside"]), ("Star", ["base_inside", "base_centred"]),
                ("Corners", ["base_a_roi_corners", "base_b_beside_fold", "base_c_roi_edge_mid"])]:
    x = Bf[Bf.folder == f]
    for c in cols:
        dd = x[c] - x.base_file
        print(f"  {f:13s} n={len(x):2d} {c:20s} r={np.corrcoef(x[c], x.base_file)[0, 1]:.3f}  median |diff| {np.median(np.abs(dd)):.2f}  mean diff {dd.mean():+.2f}")
print("Main readings: Edge centred on the corners, Star inside the corners, Corners reading (a).\n")
D = diffs(T)
def summary(d, folds=FOLDS):
    g = d[d.fold.isin(folds)].groupby("fold")["diff"]
    return pd.DataFrame({"n": g.size(), "mean": g.mean(), "sem": g.sem(), "median": g.median()}).reindex(folds)
print(summary(D, FOLDS + ["Edge", "S40dark"]).round(2))

## s4-baseline-no-difference

> Absolute baseline activity did not differ significantly across the three visual stimuli
> (Square and Edge, Star, Corners).

In [ ]:
def baseline_tests(B, col):
    groups = {f: B.loc[B.folder == f, col].dropna().values for f in ["Edge-compare", "Star", "Corners"]}
    pairs = [("Edge-compare", "Star"), ("Edge-compare", "Corners"), ("Star", "Corners")]
    p = [stats.ranksums(groups[a], groups[b]).pvalue for a, b in pairs]
    pbh = multipletests(p, method="fdr_bh")[1]
    out = {f"rank sum BH {a[:4]}-{b[:4]}": q for (a, b), q in zip(pairs, pbh)}
    out["Kruskal-Wallis p"] = stats.kruskal(*groups.values()).pvalue
    out["ANOVA p"] = stats.f_oneway(*groups.values()).pvalue
    return out, groups

B = T.drop_duplicates("code").copy()
res = {}
r_, g_ = baseline_tests(B, "base_main"); res["main readings"] = r_
for f, v in g_.items():
    q = np.percentile(v, [0, 25, 50, 75, 100])
    print(f"{f:13s} n={len(v):2d} min {q[0]:.2f} Q1 {q[1]:.2f} median {q[2]:.2f} Q3 {q[3]:.2f} max {q[4]:.2f} spikes/s")
res["zero pixels left out"] = baseline_tests(B, "base_nz")[0]
res["files' BaseActivity (38 neurons)"] = baseline_tests(B, "base_file")[0]
for name, kw in VARIANTS.items():
    res[name] = baseline_tests(with_base(B, **kw), "base_main")[0]
print()
print(pd.DataFrame(res).T.round(4))
print("(columns: rank sum tests with Benjamini-Hochberg correction for each pair, Kruskal-Wallis and ANOVA p-values)")

**Result: not reproduced (resting on the baseline placement).** Under the main readings the
Corners baseline is lower than the Square and Edge baseline (medians 2.45 and 7.11 spikes/s;
rank sum with Benjamini-Hochberg correction p = 0.044), and the Kruskal-Wallis (p = 0.028) and
one-way ANOVA (p = 0.0052) tests also find a difference, so all three readings of the criterion
find one. The outcome depends on where the baseline squares are: with the files' own BaseActivity
(38 neurons) none of the tests finds a difference (smallest p = 0.11), nor does any with Corners
readings (b) or (c) (smallest p 0.056 to 0.077), while the other placements find differences. The Star squares fall partly where the maps are
zero (39% of their pixels), which lowers the Star baseline (median 1.89 spikes/s).

## fig4a-above-baseline

> For every fold (P180, S90, S52, S40), most neurons' midpoint fold responses lie above their
> absolute baseline activity, and the mass center of each fold's points lies above the line of
> unit slope, with the mass center's distance from that line increasing as the fold angle
> decreases.

In [ ]:
def fig4a(d):
    rows = []
    for f in FOLDS:
        x = d[d.fold == f]
        pos = (x.resp > 0) & (x.base > 0)
        rows.append(dict(fold=f, n=len(x), above=int((x.resp > x.base).sum()), frac_above=(x.resp > x.base).mean(),
                         dist_raw=x.resp.mean() - x.base.mean(),
                         dist_log=np.log10(x.resp[pos]).mean() - np.log10(x.base[pos]).mean(), n_log=int(pos.sum())))
    t = pd.DataFrame(rows).set_index("fold")
    return t
for name, d in [("main", D), ("baseline without zero pixels", diffs(T, "base_nz")), ("unsmoothed maps", diffs(TU))] + \
               [(k, diffs(with_base(T, **kw))) for k, kw in VARIANTS.items()]:
    t = fig4a(d)
    print(name); print(t.round(3))
    for col in ["dist_raw", "dist_log"]:
        v = t[col].values
        print(f"  {col}: ordered P180 < S90 < S52 < S40: {bool(np.all(np.diff(v) > 0))}")
    print()

fig, ax = plt.subplots(1, 4, figsize=(14, 3.4))
for a, f in zip(ax, FOLDS):
    x = D[D.fold == f]
    a.loglog(x.base.clip(lower=0.05), x.resp.clip(lower=0.05), "o", ms=4)
    a.plot([0.05, 200], [0.05, 200], "k:")
    a.plot(x.base.mean(), x.resp.mean(), "r+", ms=14)
    a.set_title(f"{f} (n={len(x)})"); a.set_xlabel("baseline (spikes/s)")
ax[0].set_ylabel("midpoint response (spikes/s)")
plt.tight_layout(); plt.show()
print("values below 0.05 spikes/s are drawn at 0.05:", int(((D.base < 0.05) | (D.resp < 0.05))[D.fold.isin(FOLDS)].sum()), "points")

**Result: reproduced.** For every fold more than half the neurons have their midpoint response
above baseline (19 of 22 for P180, 20 of 22 for S90, 12 of 12 for S52, 26 of 26 for S40), every
mass center lies above the unit line, and the vertical distances are ordered P180 < S90 < S52 <
S40 under both readings (raw means 4.62, 8.84, 19.67, 26.32 spikes/s; log ratios 0.238, 0.357,
0.946, 1.037). The order under reading (a) holds under every baseline variant printed; under
reading (b) it fails with Star squares centred on the corners or Corners readings (b) and (c).

## fig4b-spike-rate-difference

> The mean spike rate difference at the fold midpoint decreases with increasing fold angle, from
> S40 (25.60 spikes/s) through S52 (18.10) and S90 (10.69) to P180 (4.87).

In [ ]:
EXP = {"S40": (25.60, 4.18), "S52": (18.10, 2.39), "S90": (10.69, 1.94), "P180": (4.87, 1.15)}
def fig4b(d, label):
    s = summary(d)
    s["expected"] = [EXP[f][0] for f in s.index]
    s["tol"] = [EXP[f][1] for f in s.index]
    s["within"] = (s["mean"] - s["expected"]).abs() <= s["tol"]
    order = bool(s.loc["S40", "mean"] > s.loc["S52", "mean"] > s.loc["S90", "mean"] > s.loc["P180", "mean"])
    print(f"{label}: order S40 > S52 > S90 > P180: {order}; within tolerance: {int(s.within.sum())} of 4")
    print(s.round(2)); print()
    return s
s_main = fig4b(D, "main (smoothed maps, main baselines)")
fig4b(diffs(TU), "unsmoothed maps")
fig4b(diffs(T, "base_nz"), "baselines without zero pixels")
for name, kw in VARIANTS.items():
    fig4b(diffs(with_base(T, **kw)), name)
dd = diffs(T, "base_file"); dd = dd[dd.base.notna()]
fig4b(dd, "files' BaseActivity, neurons that have it")
print("By monkey (main):")
print(D[D.fold.isin(FOLDS)].groupby(["fold", "monkey"])["diff"].agg(["size", "mean", "sem"]).round(2))

**Result: reproduced (resting on the baseline placement).** The mean midpoint spike rate
differences are S40 26.32, S52 19.67, S90 8.84 and P180 4.62 spikes/s, ordered as in the paper and
each within the paper's SEM of 25.60, 18.10, 10.69 and 4.87. The unsmoothed maps also give 4 of 4.
The order holds under every variant printed, but under the other baseline placements one value
falls outside its tolerance (3 of 4 within; Corners reading (c) gives 4 of 4). The data hold 26
Corners and 12 Star neurons in place of 28 and 30.

## fig4b-pairwise-significance

> The midpoint spike rate difference for S40 is significantly larger than for S90 and for P180,
> and that for S52 is significantly larger than for P180.

In [ ]:
import itertools
def pairwise(d):
    pairs = list(itertools.combinations(["S40", "S52", "S90", "P180"], 2))
    rows = []
    for a, b in pairs:
        xa, xb = d.loc[d.fold == a, "diff"].dropna(), d.loc[d.fold == b, "diff"].dropna()
        rows.append(dict(pair=f"{a}>{b}", median_diff=xa.median() - xb.median(), p=stats.ranksums(xa, xb).pvalue))
    t = pd.DataFrame(rows).set_index("pair")
    t["p_BH_all6"] = multipletests(t.p, method="fdr_bh")[1]
    named = ["S40>S90", "S40>P180", "S52>P180"]
    t["p_BH_named3"] = np.nan
    t.loc[named, "p_BH_named3"] = multipletests(t.loc[named, "p"], method="fdr_bh")[1]
    return t, named
for label, d in [("main", D), ("unsmoothed", diffs(TU)), ("baselines without zero pixels", diffs(T, "base_nz"))] + \
               [(k, diffs(with_base(T, **kw))) for k, kw in VARIANTS.items()]:
    t, named = pairwise(d)
    print(label); print(t.round(4))
    for col in ["p_BH_all6", "p_BH_named3"]:
        ok = all((t.loc[p, col] < 0.05) and (t.loc[p, "median_diff"] > 0) for p in named)
        print(f"  {col}: all three named pairs significant in the stated direction: {ok}")
    print()

**Result: reproduced.** S40 is larger than S90 (corrected p = 0.0003 over six pairs) and than
P180 (p < 0.0001), and S52 is larger than P180 (p = 0.0001), and the three stay significant when
the family is the three named pairs. S52 against S90 is also significant (p = 0.0074); S40 against
S52 (p = 0.43) and S90 against P180 (p = 0.12) are not. All baseline variants and the unsmoothed
maps give the same for the three named pairs.

## fig5-average-maps

> The population-average spike rate difference maps show neural activity increasing as the corner
> angle decreases.

In [ ]:
avg = {}
for f in ["Edge-compare", "Star", "Corners"]:
    rs = [r for r in recs if r["folder"] == f and r["has_standard"]]
    base = T.drop_duplicates("code").set_index("code").base_main
    ref = max(rs, key=lambda r: r["S"].size)          # largest frame; 6 Corners frames are cropped by 8 DU
    k = list(ref["vert"])[0]
    stack = []
    for r in rs:
        dx, dy = np.round(ref["vert"][k][99].mean(0) - r["vert"][k][99].mean(0)).astype(int) if f != "Corners" else np.round(ref["vert"][k][50][0] - r["vert"][k][50][0]).astype(int)
        c = np.full(ref["S"].shape, np.nan)
        h, w = r["S"].shape
        c[dy:dy + h, dx:dx + w] = r["S"] - base[r["code"]]
        stack.append(c)
    avg[f] = (np.nanmean(stack, axis=0), ref, len(rs))
rows = []
for f, (M, r0, n) in avg.items():
    g = pvc12.fold_geometry(r0)
    for fd in ("Edge", "P180", "S90", "S52", "S40"):
        if fd in g:
            rows.append(dict(fold=fd, n=n, midpoint=pvc12.midpoint(M, g[fd])[0],
                             mean_of_blocks=np.mean([pvc12.blocks(M, h) for h in g[fd]], 0).mean()))
A = pd.DataFrame(rows).set_index("fold")
print(A.round(2))
for col in ["midpoint", "mean_of_blocks"]:
    v = A.loc[["S40", "S52", "S90", "P180"], col].values
    print(f"{col}: order S40 > S52 > S90 > P180: {bool(np.all(np.diff(v) < 0))}")
fig, ax = plt.subplots(1, 3, figsize=(14, 5))
lim = max(np.nanpercentile(np.abs(M), 99) for M, _, _ in avg.values())
for a, (f, (M, r0, n)) in zip(ax, avg.items()):
    im = a.imshow(M, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim)
    a.set_title(f"{pvc12.COND[f]} (N={n})")
    g = pvc12.fold_geometry(r0)
    for fd in ("Edge", "P180", "S90", "S52", "S40"):
        for h in g.get(fd, []):
            m = (h["p0"] + h["p100"]) / 2; a.plot(m[0], m[1], "k+", ms=10)
plt.colorbar(im, ax=ax, shrink=0.7, label="spike rate difference (spikes/s)"); plt.show()
print(f"colour range +-{lim:.1f} spikes/s; values outside: " +
      ", ".join(f"{f} {int((np.abs(M) > lim).sum())} px" for f, (M, _, _) in avg.items()))

**Result: reproduced.** On the average spike rate difference maps the midpoint values are S40
26.32, S52 19.73, S90 8.74 and P180 4.54 spikes/s, ordered as the criterion requires; as the
criterion says, this repeats the means of fig4b-spike-rate-difference and is not an independent
test. The block-mean variant gives the same order (23.03, 17.32, 7.85, 3.84).

## Table 1 (table1-edge, table1-p180, table1-s90, table1-s52, table1-s40)

Per fold, the ten 10% luminance blocks (8 DU wide) give per neuron a minimum, maximum, mean and
median spike rate difference. Reading (a): per neuron summaries, then mean over neurons; reading
(b): summaries of the population-average block profile. For S90 and S52, (i) the two halves
averaged block by block; (ii) each half summarised, then the two summaries averaged.

In [ ]:
TAB = {"Edge": [7.98, 33.13, 20.44, 20.77, 22.01], "P180": [0.18, 10.28, 4.55, 4.03, 4.87],
       "S90": [4.36, 14.61, 9.22, 9.31, 10.69], "S52": [6.26, 25.39, 15.66, 15.93, 18.10],
       "S40": [9.33, 35.11, 22.81, 22.83, 25.60]}
TOL = {"Edge": [1.60, 5.11, 3.33, 3.48, 3.41], "P180": [1.01, 1.53, 1.12, 1.06, 1.15],
       "S90": [1.07, 2.48, 1.61, 1.62, 1.94], "S52": [1.33, 3.52, 2.23, 2.36, 2.39],
       "S40": [1.69, 4.17, 3.11, 3.48, 4.18]}
STATS = ["min", "max", "mean", "median", "midpoint"]
def table1(d, fold):
    x = d[d.fold == fold]
    base = x.base.values[:, None]
    out = {}
    readings = {"i": [np.vstack(x.blocks.values) - base]}
    if fold in ("S90", "S52"):
        readings["ii"] = [np.vstack([b[k] for b in x.blocks_halves.values]) - base for k in (0, 1)]
    for rk, mats in readings.items():
        # (a) per neuron then mean over neurons
        per = [np.c_[m.min(1), m.max(1), m.mean(1), np.median(m, 1)] for m in mats]
        per = np.mean(per, axis=0)
        a = list(per.mean(0)) + [x["diff"].mean()]
        a_sem = list(per.std(0, ddof=1) / np.sqrt(len(per))) + [x["diff"].sem()]
        # (b) summaries of the population-average profile
        prof = [m.mean(0) for m in mats]
        b = list(np.mean([[p.min(), p.max(), p.mean(), np.median(p)] for p in prof], axis=0)) + [x["diff"].mean()]
        out[f"{rk}a"] = a; out[f"{rk}a_sem"] = a_sem; out[f"{rk}b"] = b
    t = pd.DataFrame(out, index=STATS)
    t["paper"] = TAB[fold]; t["tol"] = TOL[fold]
    for c in [c for c in out if not c.endswith("sem")]:
        t[f"in_{c}"] = (t[c] - t.paper).abs() <= t.tol
    return t
def judge(t):
    best = max([c for c in t.columns if c.startswith("in_")], key=lambda c: t[c].sum())
    n = int(t[best].sum())
    return ("reproduced" if n == 5 else "partially_reproduced" if n >= 3 else "not_reproduced"), best, n
T1 = {}
for fold in ["Edge", "P180", "S90", "S52", "S40"]:
    t = table1(D, fold); T1[fold] = t
    print(f"== {fold} (n={int((D.fold == fold).sum())})"); print(t.round(2))
    print("  judged:", judge(t)); print()
print("Unsmoothed maps, reading (i)(a), values within tolerance per fold:",
      {f: int(table1(diffs(TU), f)["in_ia"].sum()) for f in TAB})
print("Baselines without zero pixels, best reading, within tolerance per fold:",
      {f: judge(table1(diffs(T, "base_nz"), f))[2] for f in TAB})
for name, kw in VARIANTS.items():
    print(f"{name}: values within tolerance under the best reading, per fold:",
          {f: judge(table1(diffs(with_base(T, **kw)), f))[2] for f in TAB})

**Result: S40 reproduced; Edge, P180, S90 and S52 partially reproduced.** Under reading (i)(a)
(per neuron summaries, halves averaged) S40 has all five values within the paper's SEM, and Edge,
P180, S90 and S52 have four of five each. The value outside is the minimum block for Edge (6.13
against 7.98 ± 1.60), P180 (-1.29 against 0.18 ± 1.01) and S90 (2.96 against 4.36 ± 1.07), and the
minimum for S52 (9.15 against 6.26 ± 1.33). Reading (b), summaries of the average profile, matches
fewer values for every fold, as its minimum and maximum are compressed. The counts depend on the
baseline placement (variants printed): with Edge squares inside the corners, Edge, P180 and S90
fall to 1 or 2 of 5.

## fig6-midpoint-representative

> For each fold and the Edge, the spike rate difference at the midpoint does not differ
> significantly from the mean or the median spike rate difference along the fold or Edge.

Wilcoxon rank sum tests at alpha 0.05, no correction, of per-neuron midpoint spike rate differences
against per-neuron means and medians of the ten blocks (reading (i), halves averaged), with the
signed rank test as a variant.

In [ ]:
rows = []
for fold in ["Edge", "P180", "S90", "S52", "S40"]:
    x = D[D.fold == fold]
    m = np.vstack(x.blocks.values) - x.base.values[:, None]
    for nm, v in [("mean", m.mean(1)), ("median", np.median(m, 1))]:
        rows.append(dict(fold=fold, against=nm, n=len(x), median_mid_minus=np.median(x["diff"].values - v),
                         p_ranksum=stats.ranksums(x["diff"].values, v).pvalue,
                         p_signedrank=stats.wilcoxon(x["diff"].values, v).pvalue))
F6 = pd.DataFrame(rows)
print(F6.round(4))
print("\nrank sum tests significant at 0.05:", int((F6.p_ranksum < 0.05).sum()), "of", len(F6))
print("signed rank tests significant at 0.05 (variant):", int((F6.p_signedrank < 0.05).sum()), "of", len(F6))

**Result: reproduced.** None of the ten rank sum tests is significant at 0.05 (smallest p =
0.45). The paired signed rank variant finds the midpoint above the block median for S90 (p =
0.028) and above the block mean and median for S40 (p = 0.027 and 0.0094): the midpoints sit a
little above the fold averages (median differences 0.06 to 1.9 spikes/s), which the paper's
unpaired test does not detect.

## fig7a-rms-contrast-fit

> Local RMS contrast along the Edge rises from 0.61 to 0.943 with luminance and is fit by an
> exponential model, f(l) = -0.34 exp(-0.05 l) + 0.95, with R2 = 0.9983.

Computed from each Edge neuron's stimulus image, warped into the ROI frame: a 16 x 16 DU square
centred on the Edge, moved from 0% to 100% in 1.5% steps. Readings: (a) SD of pixel luminance over
its mean; (b) SD of luminance normalised to 0 to 1. Fit f(l) = a exp(-b l) + c by least absolute
residuals (MATLAB's LAR), started from a Levenberg-Marquardt least-squares fit.

In [ ]:
def fit_exp(l, y):
    f = lambda p, l: p[0] * np.exp(-p[1] * l) + p[2]
    p0 = optimize.least_squares(lambda p: f(p, l) - y, [-0.3, 0.05, y[-1]], method="lm").x
    p = optimize.minimize(lambda p: np.abs(f(p, l) - y).sum(), p0, method="Nelder-Mead",
                          options=dict(xatol=1e-8, fatol=1e-10, maxiter=20000)).x
    r2 = 1 - ((y - f(p, l)) ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return p, r2
prof = {"a": [], "b": []}
for r in recs:
    if r["folder"] == "Edge-compare" and r["has_standard"]:
        e = pvc12.fold_geometry(r)["Edge"][0]
        lv, vals = pvc12.rms_contrast_profile(r["stim_roi"], e["p0"], e["p100"])
        prof["a"].append(vals.std(1) / vals.mean(1))
        prof["b"].append((vals / 100).std(1))
EXP7A = [0.61, 0.943, -0.34, 0.05, 0.95, 0.9983]; TOL7A = [0.061, 0.094, 0.034, 0.005, 0.095, 0.0017]
rows = {}
for k in ("a", "b"):
    P = np.array(prof[k]); y = np.median(P, 0)
    p, r2 = fit_exp(lv, y)
    rows[k] = [y.min(), y.max(), p[0], p[1], p[2], r2]
    print(f"reading ({k}): across {len(P)} stimuli, value at 0%: {P[:, 0].min():.3f}-{P[:, 0].max():.3f}, "
          f"at 50%: {np.median(P[:, np.argmin(np.abs(lv - 50))]):.3f}, at 100%: {P[:, -1].min():.3f}-{P[:, -1].max():.3f}")
t = pd.DataFrame(rows, index=["lowest", "highest", "a", "b", "c", "R2"])
t["paper"] = EXP7A; t["tol"] = TOL7A
for k in ("a", "b"):
    t[f"in_{k}"] = (t[k] - t.paper).abs() <= t.tol
print(t.round(4))
plt.figure(figsize=(5, 3.5))
for k in ("a", "b"):
    plt.plot(lv, np.median(prof[k], 0), ".", label=f"reading ({k})")
plt.plot(lv, -0.34 * np.exp(-0.05 * lv) + 0.95, "k-", label="paper's fit")
plt.xlabel("Edge luminance (%)"); plt.ylabel("local contrast"); plt.legend(); plt.show()
print("plotted range: reading (a)", np.round([np.min(prof['a']), np.max(prof['a'])], 3), "reading (b)", np.round([np.min(prof['b']), np.max(prof['b'])], 3))

**Result: not reproduced.** Neither reading gives a contrast rising from 0.61 to 0.943. Reading
(a), SD over mean, is about 2 at the 0% end (where the 16 DU square holds mostly black), falls to
0.75 at 50% and rises to 1.1 to 1.4 at 100%, so the exponential fit fails (R2 = -0.08). Reading
(b) rises from 0.013 to 0.45 and is well fit (R2 = 0.9994), but its range and parameters are far
from the paper's. The paper's definition of RMS contrast (its reference 21) is not in the text
read, so this is likely a difference of definition.

## fig7c-fold-edge-equivalence

> Neural responses to P180 are roughly equivalent to those to a 50% contrast Edge, while
> responses to S40 are close to those to a 90% contrast Edge.

Edge responses: spike rate difference in a 16 x 16 DU square centred on the Edge. Reading (a):
Edge luminance levels 0 to 100% in 10% steps; reading (b): the ten RMS contrasts 0.610 to 0.943
(step 0.037), mapped to luminance by the inverse of the paper's fit (and, as variant, of no refit,
since the refit of fig7a does not reproduce).

In [ ]:
E = D[D.fold == "Edge"].copy()
prof = np.vstack(E.edge_profile.values) - E.base.values[:, None]
mean_prof = prof.mean(0)
rms_levels = 0.610 + 0.037 * np.arange(10)
lum_of = lambda f: -np.log((f - 0.95) / -0.34) / 0.05
rms_lum = lum_of(rms_levels)
edge_rms = np.array([[np.interp(l, np.arange(101), p) for l in rms_lum] for p in prof])
print("RMS level -> luminance (%):", dict(zip(rms_levels.round(3), rms_lum.round(1))))
print("Edge mean response at 10% luminance steps:", dict(zip(range(0, 101, 10), mean_prof[::10].round(2))))
print("Edge mean response at the RMS levels:", dict(zip(rms_levels.round(3), edge_rms.mean(0).round(2))))
m = s_main["mean"]
for fold, target in [("P180", 50), ("S40", 90)]:
    la = np.arange(0, 101, 10)[np.argmin(np.abs(mean_prof[::10] - m[fold]))]
    lb = (rms_levels * 100)[np.argmin(np.abs(edge_rms.mean(0) - m[fold]))]
    if fold == "P180":
        ok_b = np.isclose(lb, 61.0)
    else:
        ok_b = abs(lb - target) <= 3.7 + 1e-9
    print(f"{fold} mean {m[fold]:.2f}: reading (a) nearest Edge level {la}% (within 10 of {target}: {abs(la - target) <= 10}); "
          f"reading (b) nearest RMS level {lb:.1f} (criterion met: {bool(ok_b)})")

**Result: not reproduced.** Under reading (a) (Edge luminance in 10% steps) the S40 mean (26.32
spikes/s) is nearest the 90% Edge level, as the paper says, but the P180 mean (4.62) is nearest 0%,
not 50%: every Edge level gives a larger response (9.04 at 0% to 20.86 at 90%). Under reading (b)
(RMS levels) P180 is nearest the lowest level, which the criterion accepts, but S40 is nearest
94.3, 4.3 from 90 and so more than one step (3.7). No single reading meets both parts.

## fig7c-fold-edge-significance

> Responses to P180 are significantly lower than responses to a 94% contrast Edge, and responses
> to S40 are significantly higher than responses to an Edge of less than 68% contrast.

In [ ]:
rows = []
for fold in FOLDS:
    xf = D.loc[D.fold == fold, "diff"].values
    for k, lvl in enumerate(rms_levels):
        tt = stats.ttest_ind(xf, edge_rms[:, k])
        rows.append(dict(fold=fold, rms=round(lvl, 3), mean_diff=xf.mean() - edge_rms[:, k].mean(), t=tt.statistic, p=tt.pvalue))
S2 = pd.DataFrame(rows)
S2["BH_all40"] = multipletests(S2.p, method="fdr_bh")[1]
S2["BH_per_fold"] = S2.groupby("fold").p.transform(lambda p: multipletests(p, method="fdr_bh")[1])
S2["Bonf_all40"] = np.minimum(S2.p * 40, 1)
S2["Bonf_per_fold"] = np.minimum(S2.p * 10, 1)
print(S2[S2.fold.isin(["P180", "S40"])].round(4).to_string())
for col in ["p", "BH_all40", "BH_per_fold", "Bonf_all40", "Bonf_per_fold"]:
    p180 = S2[(S2.fold == "P180") & (S2.rms == 0.943)].iloc[0]
    s40 = S2[(S2.fold == "S40") & (S2.rms.isin([0.61, 0.647]))]
    ok = (p180[col] < 0.05 and p180.mean_diff < 0) and bool(((s40[col] < 0.05) & (s40.mean_diff > 0)).all())
    print(f"{col:14s}: P180 < Edge(0.943) p={p180[col]:.4f}; S40 > Edge(0.610, 0.647) p={s40[col].round(4).tolist()}; criterion met: {ok}")

**Result: reproduced.** P180 is lower than the Edge at RMS 0.943 (mean difference -15.9
spikes/s, t-test p = 0.0007) and S40 is higher than the Edge at 0.610 and 0.647 (17.3 and 16.8
spikes/s, p = 0.0008 and 0.0011). Both parts hold after every correction (largest corrected p =
0.045 under Bonferroni over 40 tests). S40 is also higher than the Edge at 0.684 to 0.795 under
Benjamini-Hochberg over 40 tests, beyond the paper's "less than 68%".

## fig7d-nonlinear

> The relationship between corner angle and V1 spike rate difference is parametric but apparently
> not linear.

In [ ]:
ang = np.array([ANGLE[f] for f in FOLDS], float)
mu = s_main.loc[FOLDS, "mean"].values; se = s_main.loc[FOLDS, "sem"].values
coef = np.polyfit(ang, mu, 1); resid = mu - np.polyval(coef, ang)
print("line fit: slope %.4f, intercept %.2f" % tuple(coef))
print(pd.DataFrame({"angle": ang, "mean": mu, "sem": se, "residual": resid, "|resid|>sem": np.abs(resid) > se}, index=FOLDS).round(3))
print("monotonic (S40 > S52 > S90 > P180):", bool(np.all(np.diff(mu) > 0)))
sl = np.diff(mu) / np.diff(ang); sl_se = np.sqrt(se[1:] ** 2 + se[:-1] ** 2) / np.abs(np.diff(ang))
print("variant, successive slopes (spikes/s per degree):", dict(zip(["P180-S90", "S90-S52", "S52-S40"], sl.round(4))),
      "propagated SE:", sl_se.round(4))
for i, j in [(0, 1), (1, 2), (0, 2)]:
    print(f"  slopes {i} vs {j}: difference {abs(sl[i] - sl[j]):.4f}, combined SE {np.hypot(sl_se[i], sl_se[j]):.4f}")

**Result: reproduced.** The means fall monotonically with fold angle, and a straight line fit to
the four means leaves three of them off the line by more than their SEM (P180 +2.22 against SEM
1.03, S90 -6.09 against 2.04, S40 +4.42 against 4.18). In the variant, the slope from P180 to S90
(-0.047 spikes/s per degree) differs from that from S90 to S52 (-0.285) by 2.4 combined SE; the
S52 to S40 slope is too uncertain to compare.

## Blocked claims

- fig7d-dog-similar: needs the DOG model output of the paper's reference 3, which is not in the
  data set and is not deposited anywhere as data (data_missing).
- s3-latency-peak: needs spike times and eye positions to remake the maps at other latencies; the
  data set holds only the maps made at 60 ms (data_missing).
- spikes-per-condition: see its section.

## Summary

| Claim | Outcome |
|---|---|
| neuron-counts | not reproduced |
| spikes-per-condition | blocked (data_missing) |
| s4-baseline-no-difference | not reproduced (rests on the baseline placement) |
| fig4a-above-baseline | reproduced (rests on the baseline placement) |
| fig4b-spike-rate-difference | reproduced (rests on the baseline placement) |
| fig4b-pairwise-significance | reproduced (rests on the baseline placement) |
| fig5-average-maps | reproduced (rests on the baseline placement) |
| table1-edge | partially reproduced (rests on the baseline placement) |
| table1-p180 | partially reproduced (rests on the baseline placement) |
| table1-s90 | partially reproduced (rests on the baseline placement) |
| table1-s52 | partially reproduced (rests on the baseline placement) |
| table1-s40 | reproduced (rests on the baseline placement) |
| fig6-midpoint-representative | reproduced (rests on the baseline placement) |
| fig7a-rms-contrast-fit | not reproduced |
| fig7c-fold-edge-equivalence | not reproduced (rests on the baseline placement) |
| fig7c-fold-edge-significance | reproduced (rests on the baseline placement) |
| fig7d-nonlinear | reproduced (rests on the baseline placement) |
| fig7d-dog-similar | blocked (data_missing) |
| s3-latency-peak | blocked (data_missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- It uses the maps the authors registered and smoothed (`Standard`), not maps rebuilt from spike
  times and eye positions, which the data set does not hold.
- It has 22 Square and Edge, 12 Star and 26 Corners neurons, against 25, 30 and 28. The 15 Star
  neurons without a registered map cannot be placed on the stimulus, as both the receptive field
  offset and the diagonal aligned with the neuron's orientation are missing for them.
- The baseline squares are placed by assumption, choosing for each stimulus the placement closest
  to the files' own BaseActivity; several outcomes change with the placement (shown under each
  claim).
- RMS contrast is computed under the two definitions the claims file names; the paper's is not in
  the text read.

What was found about the data set:

- It holds 53 neurons, not the 58 the paper analysed, and fewer per condition (22, 27 and 26
  against 25, 30 and 28).
- `dat.image` is a spike rate in spikes per ms, not a raw spike count as the descriptor says; only
  one neuron has its eye time map, so spike counts cannot be recovered.
- The two `ABS` files in the `Star` folder hold Corners stimuli and recordings.
- The registered `Standard` maps, their stimulus polygons and `BaseActivity` are not described;
  15 Star neurons lack them, and `BaseActivity` (38 neurons) matches no documented rule.
- Six Corners registered maps use a frame 16 DU smaller than the other twenty.
- The screen maps are in gaze coordinates, and the receptive field offsets and preferred
  orientations needed to register them are not given.

In [ ]:
print(f"total run time {time.time() - t0:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024:.0f} MB")